In [ ]:
import pandas as pd
import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "s3://lab/PESSD/csv"

files = fs.glob(f"{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/csv/")))

In [8]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


In [21]:
BI = df[(df["audio_file"]=="BI_wav.wav") & ((df["start"]<(15*60))|((df["start"]>(17.5*60))&((df["stop"]<(89*60))))|(df["start"]>(92.5*60)))]

In [23]:
BIF = df[(df["audio_file"]=="BIF_wav.wav") & ((df["start"]<(12.5*60))|((df["start"]>(14.5*60))&((df["stop"]<(76*60)))))]

In [25]:
BMF = df[(df["audio_file"]=="BMF_wav.wav") & ((df["start"]>(1.5*60)))]

In [26]:
BMH = df[(df["audio_file"]=="BMH_wav.wav")]

In [28]:
SCR = df[(df["audio_file"]=="SCR2_wav.wav") & ((df["start"]<(28*60)))]

In [29]:
d = pd.concat([BI,BIF,BMH,BMF,SCR])

In [37]:
!pip install bertopic sentence-transformers umap-learn hdbscan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 44.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 73.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 47.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 79.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 75.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 442.0/915.6 MB 86.0 MB/s eta 0:00:06

In [1]:
import pandas as pd
d = pd.read_csv("data/test.csv")

In [2]:
import spacy
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
d['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(d['text'])]

In [7]:
groups = (d["main_g"] != d["main_g"].shift()).cumsum()

# merge consecutive texts
interventions = (
    d.groupby(groups)
      .agg({
          "main_g": "first",
          "text": " ".join
      })
      .reset_index(drop=True)
)

In [9]:
d

,Unnamed: 0,start,stop,text,main_speaker,main_g,audio_file,lemmatized_text
0,0,0.00,61.20,"En endroit, on est parti pour vivre encore un ...",SPEAKER_08,male,BI_wav.wav,"en endroit , on être partir pour vivre encore ..."
1,1,61.20,103.60,"Oui, bonjour à tous, effectivement. Bienvenue ...",SPEAKER_02,male,BI_wav.wav,"Oui , bonjour à tout , effectivement . bienven..."
2,2,103.60,125.60,On imagine un Quentin qui aura envie de conser...,SPEAKER_12,male,BI_wav.wav,on imaginer un quentin qui avoir envier de con...
3,3,125.60,146.60,Moi je vais vous donner les numéros de Dossard...,SPEAKER_05,female,BI_wav.wav,Moi je aller vous donner le numéro de Dossard ...
4,4,146.60,150.60,Ouais l'équivalent à peu près de deux tours et...,SPEAKER_02,male,BI_wav.wav,Ouais le équivaloir à peu près de deux tour et...
...,...,...,...,...,...,...,...,...
1472,126,1611.68,1623.00,C'était raté sur la dernière course. La FP 4e ...,SPEAKER_11,male,SCR2_wav.wav,ce être rater sur le dernier course . le FP 4e...
1473,127,1623.16,1633.56,"La Suède, qui revient à 3 secondes dès Nord-Vé...",SPEAKER_06,male,SCR2_wav.wav,"le Suède , qui revenir à 3 seconde dès Nord-Vé..."
1474,128,1634.88,1640.40,Ça va pas le mettre en confiance pour le début...,SPEAKER_10,male,SCR2_wav.wav,cela aller pas le mettre en confiance pour le ...
1475,129,1640.56,1645.64,C'était déjà la catastrophe à Nové-Mesto. Il a...,SPEAKER_06,female,SCR2_wav.wav,ce être déjà le catastrophe à Nové - Mesto . i...


In [16]:
# Install dependencies if needed
# pip install bertopic sentence-transformers umap-learn hdbscan

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
)

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="french",
    calculate_probabilities=True,
    verbose=True
)

# Fit the model
topics, probs = topic_model.fit_transform(d["text"])

# Show discovered topics
print(topic_model.get_topic_info())

# Show keywords for a specific topic
print(topic_model.get_topic(0))

# Visualize topics
topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

#keybertopic, extrait phrases saillantes
#tester lda
#virer phrases nulles et refaire tourner
#pas virer -1 sauf si faible échantillon et checker
#checker si semi-superviser pour trouver thèmes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5877.77it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-13 16:53:04,853 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 47/47 [18:03<00:00, 23.05s/it]
2026-03-13 17:11:08,388 - BERTopic - Embedding - Completed ✓
2026-03-13 17:11:08,389 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-13 17:12:08,374 - BERTopic - Dimensionality - Completed ✓
2026-03-13 17:12:08,378 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-13 17:12:08,520 - BERTopic - Cluster - Completed ✓
2026-03-13 17:12:08,526 - BERTopic - Representation - Fine-tuning topics using representation 

    Topic  Count                                               Name  \
0      -1    584                       -1_tir_course_julia_secondes   
1       0    199                 0_julia_julia simon_simon_secondes   
2       1    183              1_eric_perrault_eric perrault_quentin   
3       2     57                 2_secondes_intermédiaire_repris_10   
4       3     55                            3_tir_balle_balles_posé   
5       4     39        4_applaudissements_allez_pioche_allez allez   
6       5     35                5_grave_allez lou_dernière_retourne   
7       6     33               6_relance_quentin_allez quentin_fini   
8       7     30      7_fillon_quentin fillon_quentin_philippe horn   
9       8     28                  8_course_tour_française_italienne   
10      9     25                                  9_20_20 20_19_000   
11     10     24                     10_faut_vraiment_piste_pousser   
12     11     22                          11_skis_ski_bosse_relance   
13    

In [18]:
t = pd.DataFrame({
    "text": d['text'],
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})

In [20]:
t["topic"].value_counts()

topic
-1     584
 0     199
 1     183
 2      57
 3      55
 4      39
 5      35
 6      33
 7      30
 8      28
 9      25
 10     24
 11     22
 12     20
 13     20
 14     19
 15     16
 16     15
 17     13
 18     13
 19     12
 20     12
 21     12
 22     11
Name: count, dtype: int64

In [50]:
for t in t[t["topic"]==-1]["text"] : 
    print(t)

On imagine un Quentin qui aura envie de conserver son titre, mais face à lui, la startlist est belle. Il va falloir se méfier des Norvégiens qui ont sorti la grosse équipe aujourd'hui. Et attention, on sera dans le vif du sujet tout de suite avec l'expérience de Jakov Fak, le Slovene, et Fabien Claude, qui se lancera avec le 16. Donc il va falloir être attentif tout de suite.
Moi je vais vous donner les numéros de Dossard à suivre d'abord des Français. Donc il y a le Dossard 16, c'est Fabien Claude, donc Fabien qui est un gros skieur, un des meilleurs du circuit. Donc tout l'enjeu pour lui cette année, ça va être vraiment de mettre les balles au fond parce que voilà, c'est une course où il s'est une minute de pénalité par balle manquée. Donc tout de suite...
Ouais l'équivalent à peu près de deux tours et demi de pénalité.
Ouais voilà, c'est une grosse pénalité. Après aujourd'hui c'est quand même une piste pour les gros skieurs parce qu'on vous le détaillera un peu plus tard, mais il y 

In [48]:
topic_model.get_topic(-1)

[('tir', np.float64(0.019705384158766548)),
 ('course', np.float64(0.01810100205230608)),
 ('julia', np.float64(0.016166922702569482)),
 ('secondes', np.float64(0.01581559885555621)),
 ('petit', np.float64(0.015216028128888991)),
 ('vraiment', np.float64(0.014750896397992897)),
 ('faut', np.float64(0.014264980364740047)),
 ('allez', np.float64(0.014103260774615873)),
 ('20', np.float64(0.01389915649281138)),
 ('aller', np.float64(0.013604787866193726))]

In [65]:
test = topic.merge(d, on="text")

In [67]:
test[(test["topic"]==10)|(test["topic"]==15)]

,text,topic,topic_prob,start,stop,main_speaker,main_g,audio_file
8,Pour vous en dire un petit peu plus sur la pis...,10,1.000000,289.60,345.80,SPEAKER_05,female,BI_wav.wav
9,On discutait hier justement dans la reconnaiss...,10,0.124827,345.80,356.40,SPEAKER_12,male,BI_wav.wav
11,Après c'est vrai que cette année il y a des st...,15,1.000000,374.20,386.60,SPEAKER_05,female,BI_wav.wav
13,"C'est un athlète solide, le jeune Suisse qui p...",15,0.238053,421.20,434.20,SPEAKER_12,male,BI_wav.wav
25,C'est un athlète qui a énormément d'expérience...,15,0.219121,604.20,622.20,SPEAKER_12,male,BI_wav.wav
26,"Cette année, il est un tout petit peu moins bo...",15,1.000000,622.20,634.20,SPEAKER_05,female,BI_wav.wav
28,"Fabien, c'est vraiment un skieur qui fait part...",15,1.000000,677.20,709.20,SPEAKER_05,female,BI_wav.wav
78,"Là, ça fait mal. On voit dans le regard qu'il ...",10,0.364798,1586.20,1595.20,SPEAKER_12,female,BI_wav.wav
93,On a l'impression qu'il y a les jambes qui tre...,10,1.000000,1724.20,1734.20,SPEAKER_05,female,BI_wav.wav
639,"Comme on l'avait dit pour le relais mixte, c'e...",10,0.152644,4025.84,4036.72,SPEAKER_05,female,BI_wav.wav
